# SQL query from table names - Continued

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [3]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [36]:
context = [ {'role':'system', 'content': """
You are a SQL generation assistant. Use the following database schema to answer user questions.

STRICT RULES:
- All answers must start with "this is your SQL"
- Then provide the SQL inside a ```sql code block
- After that, give a brief explanation
- Use only the tables and columns provided
- Keep the SQL simple and valid for SQLite

### Database Schema:

CREATE TABLE employees (
  ID_usr INT PRIMARY KEY,
  name VARCHAR(100)
);
/*
Sample rows:
(1, 'John Doe'),
(2, 'Jane Smith'),
(3, 'Robert Brown')
*/

CREATE TABLE salary (
  ID_usr INT,
  year DATE,
  salary FLOAT,
  FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);
/*
Sample rows:
(1, '2023-01-01', 50000.0),
(2, '2023-01-01', 65000.0),
(1, '2024-01-01', 52000.0),
(3, '2023-01-01', 47000.0)
*/

CREATE TABLE studies (
  ID INT PRIMARY KEY,
  ID_usr INT,
  educational_level INT,
  Institution VARCHAR(150),
  Years DATE,
  Speciality VARCHAR(100),
  FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);
/*
Sample rows:
(101, 1, 3, 'Stanford University', '2020-06-15', 'Computer Science'),
(102, 2, 4, 'University of Cambridge', '2018-05-20', 'Physics'),
(103, 3, 3, 'ETH Zurich', '2019-07-10', 'Business')
*/
"""} ]

In [38]:
# FEW SHOT SAMPLES
context.append({'role':'system', 'content':"""
-- Maintain the SQL order simple and efficient as you can, using valid SQLite.
-- Answer the following questions based on the tables above:

Question: Show all employee names.
SQL:
SELECT name FROM employees;

Question: Show all salaries for 2023.
SQL:
SELECT salary FROM salary WHERE year = '2023-01-01';

Question: Show employee names with their salaries.
SQL:
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr;

Question: Show employees who studied Physics.
SQL:
SELECT e.name
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
WHERE st.Speciality = 'Physics';
"""})

In [39]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [58]:
#new
context_user = context.copy()
print(return_CCRMSQL("Show all employee names", context_user))

print("----------")

#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Show all employee names", old_context_user))

this is your SQL
```sql
SELECT name FROM employees;
```
Explanation: This query selects and displays all employee names from the "employees" table.
----------
This is your SQL:
```sql
SELECT name FROM employees;
```
Explanation: This SQL command selects the "name" column from the "employees" table, showing all employee names.


In [59]:
#new
print(return_CCRMSQL("Which employee has the highest salary in 2023?", context_user))

print("----------")

#old
print(return_CCRMSQL("Which employee has the highest salary in 2023?", old_context_user))

this is your SQL
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary = (SELECT MAX(salary) FROM salary WHERE year = '2023-01-01');
```

Explanation: This SQL query retrieves the name of the employee who has the highest salary in the year 2023 by comparing the salary with the maximum salary in 2023.
----------
This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2023'
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee with the highest salary in the year 2023 by joining the "employees" and "salary" tables on the employee ID. It then filters the results to only include salaries from 2023, orders them in descending order, and limits the output to only the top result, which corresponds to the employee with the highest salary in 2023.


*Version 2*

In [25]:
context_v2 = [ {'role':'system', 'content': """
You are a SQL generation assistant.

STRICT RULES:
- Start with "this is your SQL"
- Use ```sql code block
- Add short explanation

### Database Schema:

CREATE TABLE employees (
  ID_usr INT PRIMARY KEY,
  name VARCHAR(100)
);

CREATE TABLE salary (
  ID_usr INT,
  year DATE,
  salary FLOAT
);

CREATE TABLE studies (
  ID INT PRIMARY KEY,
  ID_usr INT,
  educational_level INT,
  Institution VARCHAR(150),
  Years DATE,
  Speciality VARCHAR(100)
);
"""} ]

In [27]:
#Function to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext_v2 = context_v2.copy()
    newcontext_v2.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext_v2,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [60]:
#_v2
context_v2_user = context_v2.copy()
print(return_CCRMSQL("List all employees", context_v2_user))

print("----------")

#old
old_context_user = old_context.copy()
print(return_CCRMSQL("List all employees", old_context_user))


this is your SQL

```sql
SELECT * FROM employees;
```

Explanation: This SQL query selects all columns from the employees table, which will list all employees in the database.
----------
This is your SQL:
```sql
SELECT * FROM employees;
```
This SQL command retrieves all the records from the "employees" table, listing all employees stored in the database.


In [61]:
#_v2
context_v2_user = context_v2.copy()
print(return_CCRMSQL("Show each employee's name, institution, speciality, salary, and salary year", context_v2_user))

print("----------")

#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Show each employee's name, institution, speciality, salary, and salary year", old_context_user))


this is your SQL

```sql
SELECT e.name, s.year, st.Institution, st.Speciality, sa.salary
FROM employees e
JOIN salary sa ON e.ID_usr = sa.ID_usr
JOIN studies st ON e.ID_usr = st.ID_usr;
```

Explanation: This SQL query retrieves the name of each employee, their salary, salary year, institution, and speciality by joining the employees, salary, and studies tables on the corresponding ID columns.
----------
This is your SQL:
```sql
SELECT e.name, s.salary, s.year, st.Institution, st.Speciality
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
JOIN studies st ON e.ID_usr = st.ID_usr;
```

This SQL query retrieves the name of each employee, their salary, salary year, institution, and speciality by joining the "employees," "salary," and "studies" tables on the corresponding employee ID.


In [62]:
#_v2
print(return_CCRMSQL("Find employees who studied Computer Science and earned more than 50000 in 2023", context_v2_user))

print("----------")

#_old
print(return_CCRMSQL("Find employees who studied Computer Science and earned more than 50000 in 2023", old_context_user))


this is your SQL

```sql
SELECT e.name
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
WHERE s.Speciality = 'Computer Science' 
AND sa.salary > 50000
AND YEAR(sa.year) = 2023;
```

Explanation: This SQL query selects the names of employees who studied Computer Science and earned more than 50000 in the year 2023 by joining the employees, studies, and salary tables and applying the necessary conditions.
----------
This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
WHERE s.Speciality = 'Computer Science' AND sa.salary > 50000 AND YEAR(sa.year) = 2023;
```

This SQL query selects the names of employees who studied Computer Science and earned more than 50000 in the year 2023. It achieves this by joining the `employees`, `studies`, and `salary` tables on the corresponding ID columns, then filtering the results based on the specified conditions.


In [63]:
#_v2
print(return_CCRMSQL("List employees whose educational level is 3 or higher and order them by salary from highest to lowest", context_v2_user))

print("----------")

#old
print(return_CCRMSQL("List employees whose educational level is 3 or higher and order them by salary from highest to lowest", old_context_user))

this is your SQL

```sql
SELECT e.name
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
WHERE s.educational_level >= 3
ORDER BY sa.salary DESC;
```

Explanation: This SQL query retrieves the names of employees whose educational level is 3 or higher by joining the `employees`, `studies`, and `salary` tables. The results are then ordered by salary in descending order.
----------
This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
JOIN studies st ON e.ID_usr = st.ID_usr
WHERE st.educational_level >= 3
ORDER BY s.salary DESC;
```

This SQL query selects the names and salaries of employees whose educational level is 3 or higher by joining the "employees," "salary," and "studies" tables. It then filters the results to only include employees with an educational level of 3 or higher and orders them by salary from highest to lowest.


*Version 3*

In [45]:
context_v3 = [ {'role':'system', 'content': """
You are a SQL generation assistant.

STRICT RULES:
1. Always start with: this is your SQL
2. Always put the SQL inside a ```sql code block
3. Always add a short explanation after the SQL
4. Use only the tables and columns provided below
5. Do not invent table names or column names
6. Use valid SQLite
7. Keep SQL simple and efficient
8. Prefer ORDER BY + LIMIT instead of subqueries when finding highest or lowest values
9. Use the exact date format from the sample rows, for example '2023-01-01'

### Database Schema:

CREATE TABLE employees (
  ID_usr INT PRIMARY KEY,
  name VARCHAR(100)
);
/*
Sample rows:
(1, 'John Doe'),
(2, 'Jane Smith'),
(3, 'Robert Brown')
*/

CREATE TABLE salary (
  ID_usr INT,
  year DATE,
  salary FLOAT,
  FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);
/*
Sample rows:
(1, '2023-01-01', 50000.0),
(2, '2023-01-01', 65000.0),
(1, '2024-01-01', 52000.0),
(3, '2023-01-01', 47000.0)
*/

CREATE TABLE studies (
  ID INT PRIMARY KEY,
  ID_usr INT,
  educational_level INT,
  Institution VARCHAR(150),
  Years DATE,
  Speciality VARCHAR(100),
  FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);
/*
Sample rows:
(101, 1, 3, 'Stanford University', '2020-06-15', 'Computer Science'),
(102, 2, 4, 'University of Cambridge', '2018-05-20', 'Physics'),
(103, 3, 3, 'ETH Zurich', '2019-07-10', 'Business')
*/
"""} ]

In [46]:
context_v3.append({'role':'system', 'content':"""
-- Maintain the SQL order simple and efficient as you can, using valid SQLite.
-- Answer the following questions based on the tables above:

Question: Show all employee names.
SQL:
SELECT name FROM employees;

Question: Which employee has the highest salary in 2023?
SQL:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2023-01-01'
ORDER BY s.salary DESC
LIMIT 1;

Question: Show employee names with their institutions and specialities.
SQL:
SELECT e.name, st.Institution, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;

Question: Show employees who studied Computer Science and earned more than 50000 in 2023.
SQL:
SELECT e.name
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
WHERE st.Speciality = 'Computer Science'
AND sa.salary > 50000
AND sa.year = '2023-01-01';
"""})

In [47]:
#Function to call the model.
def return_CCRMSQL(user_message, context_v3):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext_v3 = context_v3.copy()
    newcontext_v3.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext_v3,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [54]:
#_v3
context_v3_user = context_v3.copy()
print(return_CCRMSQL("Show all employee names", context_v3_user))

print("----------")

#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Show all employee names", old_context_user))

```sql
SELECT name FROM employees;
```
This SQL query retrieves all employee names from the `employees` table.
----------
This is your SQL:
```sql
SELECT name FROM employees;
```

Explanation: This SQL command selects the "name" column from the "employees" table, showing all employee names.


In [55]:

#_v3
print(return_CCRMSQL("Which employee has the highest salary in 2023?", context_v3_user))

print("----------")

#old
print(return_CCRMSQL("Which employee has the highest salary in 2023?", old_context_user))

```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2023-01-01'
ORDER BY s.salary DESC
LIMIT 1;
```
This SQL query retrieves the name of the employee who has the highest salary in the year 2023.
----------
This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2023'
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee with the highest salary in the year 2023 by joining the "employees" and "salary" tables on the employee ID, filtering the results for the year 2023, ordering the salaries in descending order, and limiting the output to the top result.


In [57]:
#_v3
print(return_CCRMSQL("Show employee names with their institutions and specialities", context_v3_user))


print("----------")

# old
print(return_CCRMSQL("Show employee names with their institutions and specialities", old_context_user))

```sql
SELECT e.name, st.Institution, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;
```
Explanation: This SQL query retrieves employee names along with the institutions they studied at and their specialities from the 'employees' and 'studies' tables by performing an inner join on the 'ID_usr' column.
----------
This is your SQL:
```sql
SELECT e.name, s.Institution, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;
```

This SQL query selects the employee names along with their institutions and specialities by joining the "employees" table with the "studies" table on the common column "ID_usr".


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

*Report*

In this experiment, three different prompt versions were tested to evaluate how prompt design affects the quality of SQL generation. Each version was evaluated using the same natural language queries to ensure a fair comparison.

Version 1 (Schema + Few-Shot Examples)

Produced mostly correct SQL queries.
Formatting was consistent.
Sometimes generated more complex queries than necessary (e.g., subqueries instead of simpler solutions).
Showed good understanding of joins and relationships.

Conclusion:
Version 1 performed well and was reliable, but not always optimal in simplicity.

-------

Version 2 (Schema Only, No Few-Shot)


Still able to generate correct SQL queries.
More inconsistencies in:
                        alias usage.
                        date filtering ('2023' vs '2023-01-01').
Less structured outputs.
Slightly higher chance of inefficient or messy queries.

Conclusion:
Version 2 showed that schema alone is sufficient for basic SQL generation, but lacks consistency and precision without examples.

--------

Version 3 (Schema + Few-Shot + Strict Rules)


Most consistent and clean outputs.
Correct formatting (this is your SQL, code block, explanation).
Preferred simple and efficient SQL (e.g., ORDER BY + LIMIT).
Lowest error rate across all versions.

Conclusion:
Version 3 performed best due to clear guidance, structured examples, and strict constraints.

*What did I learn?*

- Few-shot examples are very important. Without examples (Version 2), the model becomes less consistent and more error-prone.
- Examples are more powerful than instructions. The model follows patterns shown in examples more than written rules.
- More context does not always mean better results. Version 1 sometimes produced overly complex queries despite having more information.
- Strict rules improve output quality.Version 3 showed that adding constraints (e.g., prefer simple SQL) leads to better and cleaner results.
- Prompt design directly controls model behavior. Small changes in the prompt significantly changed SQL structure, formatting, and accuracy.